# 深度解析: `Junction`连接件

**相关模块:** `common/junction.py`

## 目的
在构建复杂的河网或管网模型时，节点的汇合与分流是基本拓扑关系。`Junction`组件在框架中扮演着“交通枢纽”的角色，专门用于处理这种节点上的流量变化。它是实现复杂网络（如Y型、菱形、树状）的关键。

本教程将深入解析`Junction`组件的功能和用法。

此笔记本将：
1.  解释`Junction`的核心功能：**质量守恒**。
2.  演示其在**汇流（Confluence）**场景（多进一出）中的应用。
3.  演示其在**分流（Diffluence）**场景（一进多出）中的应用，包括默认的平均分配和自定义比例分配两种模式。

## 1. 环境设置

我们导入所需的控制器、Junction本身，以及一个简单的自定义组件`Source`用于提供源流量，以简化测试。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from common.base_model import BaseModelComponent
from common.controller import SimulationController
from common.junction import Junction

# --- 定义一个简单的源组件用于测试 ---
class Source(BaseModelComponent):
    def __init__(self, name: str, hydrograph: np.ndarray):
        super().__init__(name)
        self.hydrograph = hydrograph
        self.t = 0
    def step(self, inflows: dict, dt: float):
        if self.t < len(self.hydrograph):
            self.outflow = self.hydrograph[self.t]
        else:
            self.outflow = 0.0
        self.t += 1

## 2. 汇流 (多进一出) 场景

这是`Junction`最常见的用途，例如两条支流汇入干流。在这种场景下，`Junction`的核心任务是简单地将所有上游的来流相加。

我们构建一个 `SourceA` 和 `SourceB` 同时流入 `J1` 的网络。

In [ ]:
# 定义输入
inflow_A = np.array([10, 20, 30, 20, 10])
inflow_B = np.array([5, 10, 15, 10, 5])

# 创建组件
source_a = Source(name='SourceA', hydrograph=inflow_A)
source_b = Source(name='SourceB', hydrograph=inflow_B)
junction_merge = Junction(name='J1_Merge')

# 构建网络
controller = SimulationController()
controller.add_component(source_a)
controller.add_component(source_b)
controller.add_component(junction_merge)
controller.connect('SourceA', 'J1_Merge')
controller.connect('SourceB', 'J1_Merge')

# 运行并记录历史
history = []
history_generator = controller.run(num_steps=5, dt=1)
for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        current_step_history[name] = {'outflow': comp.get_outflow(), 'inflow': getattr(comp, 'total_inflow', 0)}
    history.append(current_step_history)

# --- 验证和输出 ---
print("汇流场景验证:")
print("Time | Inflow A | Inflow B | Junction Inflow | Junction Outflow")
print("-----------------------------------------------------------------")
for i in range(5):
    in_a = history[i]['SourceA']['outflow']
    in_b = history[i]['SourceB']['outflow']
    j_in = history[i]['J1_Merge']['inflow']
    j_out = history[i]['J1_Merge']['outflow']
    print(f"{i:4d} | {in_a:8.1f} | {in_b:8.1f} | {j_in:15.1f} | {j_out:16.1f}")
    # 验证质量守恒
    assert np.isclose(j_in, in_a + in_b), f"Step {i}: Mass balance failed!"
    assert np.isclose(j_out, j_in), f"Step {i}: Outflow does not equal inflow!"

print("\n验证成功! Junction正确地将所有来流相加。")

## 3. 分流 (一进多出) 场景

`Junction`也可以处理分流。此时，它的任务是将总入流量按照一定的规则分配给下游的多个分支。

### 3.1 默认模式：平均分配
如果在创建`Junction`时没有提供`split_rules`参数，它会默认将总入流量平均分配给所有下游分支。

我们构建一个 `SourceA`流入`J2_Split`，然后`J2_Split`再分流到两个虚拟的下游组件（我们只关心Junction的行为，所以下游不需要真的组件）。

In [ ]:
source_a_split = Source(name='SourceA', hydrograph=inflow_A)
junction_split_even = Junction(name='J2_Split_Even')

# 伪造两个下游组件用于连接
downstream_b = Source('BranchB', np.array([]))
downstream_c = Source('BranchC', np.array([]))

controller = SimulationController()
controller.add_component(source_a_split)
controller.add_component(junction_split_even)
controller.add_component(downstream_b)
controller.add_component(downstream_c)

controller.connect('SourceA', 'J2_Split_Even')
controller.connect('J2_Split_Even', 'BranchB')
controller.connect('J2_Split_Even', 'BranchC')

history = []
history_generator = controller.run(num_steps=5, dt=1)
for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        # 需要特殊处理来获取分流信息
        split_outflows = getattr(comp, 'outflows', {})
        current_step_history[name] = {'outflow': comp.get_outflow(), 'inflow': getattr(comp, 'total_inflow', 0), 'split_outflows': split_outflows}
    history.append(current_step_history)

print("分流场景 (平均分配) 验证:")
print("Time | Total Inflow | Out to B | Out to C")
print("------------------------------------------")
for i in range(5):
    j_in = history[i]['J2_Split_Even']['inflow']
    out_b = history[i]['J2_Split_Even']['split_outflows']['BranchB']
    out_c = history[i]['J2_Split_Even']['split_outflows']['BranchC']
    print(f"{i:4d} | {j_in:12.1f} | {out_b:8.1f} | {out_c:8.1f}")
    assert np.isclose(out_b, j_in / 2) and np.isclose(out_c, j_in / 2), "Even split failed!"
    
print("\n验证成功! Junction正确地将流量平均分配。")

### 3.2 自定义规则分配
通过在创建`Junction`时提供`split_rules`参数，我们可以实现按任意比例分流。例如，我们将70%的流量分给B分支，30%分给C分支。

In [ ]:
source_a_split = Source(name='SourceA', hydrograph=inflow_A)
split_rules = {'BranchB': 0.7, 'BranchC': 0.3}
junction_split_custom = Junction(name='J3_Split_Custom', split_rules=split_rules)

controller = SimulationController()
controller.add_component(source_a_split)
controller.add_component(junction_split_custom)
controller.add_component(downstream_b)
controller.add_component(downstream_c)

controller.connect('SourceA', 'J3_Split_Custom')
controller.connect('J3_Split_Custom', 'BranchB')
controller.connect('J3_Split_Custom', 'BranchC')

history = []
history_generator = controller.run(num_steps=5, dt=1)
for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        split_outflows = getattr(comp, 'outflows', {})
        current_step_history[name] = {'outflow': comp.get_outflow(), 'inflow': getattr(comp, 'total_inflow', 0), 'split_outflows': split_outflows}
    history.append(current_step_history)

print("分流场景 (70/30 自定义分配) 验证:")
print("Time | Total Inflow | Out to B (70%) | Out to C (30%)")
print("-----------------------------------------------------")
for i in range(5):
    j_in = history[i]['J3_Split_Custom']['inflow']
    out_b = history[i]['J3_Split_Custom']['split_outflows']['BranchB']
    out_c = history[i]['J3_Split_Custom']['split_outflows']['BranchC']
    print(f"{i:4d} | {j_in:12.1f} | {out_b:14.1f} | {out_c:14.1f}")
    assert np.isclose(out_b, j_in * 0.7) and np.isclose(out_c, j_in * 0.3), "Custom split failed!"

print("\n验证成功! Junction正确地按自定义规则分配流量。")

### 关于分流的讨论
需要注意的是，当前`Junction`的分流是基于预设的**比例**，而不是基于物理过程（如水力坡降、下游水位等）。这是一种简化的处理方式，适用于概念模型或需要强制按特定比例分流的场景（如灌溉渠系的分水）。
在一个完全基于物理的水动力模型中，分流点的流量分配会由下游河道的水力条件动态决定，这通常需要更复杂的迭代求解器来解决，类似于处理环状网络的方式。